In [44]:
import torch 
import pandas as pd
import numpy as np 
import torch.nn as nn 
from datasets import load_dataset
from datasets import Dataset as HFDataset
from torch.utils.data import Dataset,DataLoader 
from tokenizers import Tokenizer,models,trainers,pre_tokenizers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [45]:
# Load Dataset from Hugginface libarary 

dataset = load_dataset("ai4bharat/samanantar", "hi", split="train[:200000]")

print(dataset.features)
print(dataset[0])

{'idx': Value('int64'), 'src': Value('string'), 'tgt': Value('string')}
{'idx': 0, 'src': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles", 'tgt': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}


In [46]:
df = dataset.to_pandas()

# Rename columns to match rest of pipeline
df = df.rename(columns={'src': 'en', 'tgt': 'hi'})
df = df[['en', 'hi']] 

# Remove nulls and empty strings
df = df[
    df['en'].notna() & df['hi'].notna() &
    (df['en'].str.strip() != '') &
    (df['hi'].str.strip() != '')
].reset_index(drop=True)

# Remove duplicates
df = df.drop_duplicates(subset=['en', 'hi'], keep='first').reset_index(drop=True)

print(f"After cleaning: {len(df)} pairs")
print(f"\nSample pairs:")
for i in range(5):
    print(f"EN: {df.loc[i,'en']}")
    print(f"HI: {df.loc[i,'hi']}")
    print()

After cleaning: 200000 pairs

Sample pairs:
EN: However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles
HI: आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।

EN: Whosoever desires the reward of the world, with Allah is the reward of the world and of the Everlasting Life. Allah is the Hearer, the Seer.
HI: और जो शख्स (अपने आमाल का) बदला दुनिया ही में चाहता है तो ख़ुदा के पास दुनिया व आख़िरत दोनों का अज्र मौजूद है और ख़ुदा तो हर शख्स की सुनता और सबको देखता है

EN: The value of insects in the biosphere is enormous because they outnumber all other living groups in measure of species richness.
HI: जैव-मंडल में कीड़ों का मूल्य बहुत है, क्योंकि प्रजातियों की समृद्धि के मामले में उनकी संख्या अन्य जीव समूहों से ज़्यादा है।

EN: Mithali To Anchor Indian Team Against Australia in ODIs
HI: आस्ट्रेलिया के खिलाफ वनडे टी

In [47]:
MAX_CHAR_LEN = 200

# Rmove the row that is more than define size 
df = df[ 
        (df["en"].str.len() <= MAX_CHAR_LEN) &
        (df['hi'].str.len() <= MAX_CHAR_LEN)
       ].reset_index(drop=True)

# Quick distribution check
print(f"Avg English length (chars): {df['en'].str.len().mean():.1f}")
print(f"Avg Hindi length (chars):   {df['hi'].str.len().mean():.1f}")
print(f"Max English length (chars): {df['en'].str.len().max()}")
print(f"Max Hindi length (chars):   {df['hi'].str.len().max()}")

Avg English length (chars): 79.5
Avg Hindi length (chars):   76.4
Max English length (chars): 200
Max Hindi length (chars):   200


In [48]:
VOCAB_SIZE = 8000

#---For english vocab---
# Train joint BPE tokenizer
en_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
en_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Define special tokens for NMT
en_trainer = trainers.BpeTrainer(
    vocab_size = VOCAB_SIZE,
    special_tokens = ["[PAD]","[UNK]","[SOS]","[EOS]"]
)

en_tokenizer.train_from_iterator(df['en'].tolist(),trainer=en_trainer)

# ---For hindi vocab--- 
hi_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
hi_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

hi_trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[PAD]","[UNK]","[SOS]","[EOS]"]
)

hi_tokenizer.train_from_iterator(df['hi'].tolist(),trainer=hi_trainer)

print(f"English vocab size: {en_tokenizer.get_vocab_size()}")
print(f"Hindi vocab size:   {hi_tokenizer.get_vocab_size()}")







English vocab size: 8000
Hindi vocab size:   8000


In [49]:
# Saving vocab for refetch quickliy
en_tokenizer.save("/kaggle/working/en_tokenizer.json")
hi_tokenizer.save("/kaggle/working/hi_tokenizer.json")

In [50]:
df['en_len'] = df['en'].apply(lambda x: len(en_tokenizer.encode(x).ids))
df['hi_len'] = df['hi'].apply(lambda x: len(hi_tokenizer.encode(x).ids))

df = df[
    (df['en_len'] >= 4) & (df['en_len'] <= 30) &
    (df['hi_len'] >= 4) & (df['hi_len'] <= 30)
].reset_index(drop=True)

print(f"Final pairs: {len(df)}")
print(f"Avg EN tokens: {df['en_len'].mean():.1f}")
print(f"Avg HI tokens: {df['hi_len'].mean():.1f}")

Final pairs: 135491
Avg EN tokens: 14.4
Avg HI tokens: 15.3


In [51]:
# Splitint dataset to train,validation and test 
from sklearn.model_selection import train_test_split

train_val_df,test_df = train_test_split(df,test_size=0.1,random_state=42)
train_df,val_df = train_test_split(train_val_df,test_size=0.111, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train size:      {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size:       {len(test_df)}")

Train size:      108405
Validation size: 13536
Test size:       13550


In [52]:
print("=== Data Quality Check ===")
print(f"Total pairs: {len(train_df)}")
print(f"Avg EN tokens: {train_df['en_len'].mean():.1f}")
print(f"Avg HI tokens: {train_df['hi_len'].mean():.1f}")

# Show length distribution
for min_len in [1, 3, 5, 8, 10]:
    count = (train_df['en_len'] >= min_len).sum()
    pct   = count / len(train_df) * 100
    print(f"Sentences with EN >= {min_len} tokens: {count} ({pct:.1f}%)")

# Show 10 actual samples to see data quality
print("\n=== Sample pairs ===")
for i in range(10):
    print(f"EN: {train_df.loc[i,'en']}")
    print(f"HI: {train_df.loc[i,'hi']}")
    print()

=== Data Quality Check ===
Total pairs: 108405
Avg EN tokens: 14.4
Avg HI tokens: 15.3
Sentences with EN >= 1 tokens: 108405 (100.0%)
Sentences with EN >= 3 tokens: 108405 (100.0%)
Sentences with EN >= 5 tokens: 106414 (98.2%)
Sentences with EN >= 8 tokens: 91094 (84.0%)
Sentences with EN >= 10 tokens: 77328 (71.3%)

=== Sample pairs ===
EN: It is situated north of Cairo in the Nile Delta region.
HI: यह राष्ट्रीय राजधानी क़ाहिरा से उत्तर में मिस्र के उत्तरी भाग में नील नदी के नदीमुख (डॅल्टा) क्षेत्र में स्थित है।

EN: indeed, on that Day their Lord will be aware of them!
HI: निस्संदेह उनका रब उस दिन उनकी पूरी ख़बर रखता होगा

EN: Many senior Congress leaders named in this list.
HI: कांग्रेस की इस सूची में कई बड़े नाम हैं.

EN: It was likely during this period in his life that David perfected his skills as a musician.
HI: शायद इसी समय के दौरान, संगीतकार के तौर पर दाविद के हुनर में और निखार आया होगा ।

EN: Jammu and Kashmir CM Omar Abdullah at the inauguration on Sunday.
HI: जम्मू कश्मीर क

In [53]:
# Dataset class 
class TranslationDataset(Dataset):
    
    def __init__(self,df,en_tokenizer,hi_tokenizer):
        en_sos = en_tokenizer.token_to_id("[SOS]")
        en_eos = en_tokenizer.token_to_id("[EOS]")
        hi_sos = hi_tokenizer.token_to_id("[SOS]")
        hi_eos = hi_tokenizer.token_to_id("[EOS]")

        print("Pre-tokenizing English...")
        en_encoded = en_tokenizer.encode_batch(df['en'].tolist())
        print("Pre-tokenizing Hindi...")
        hi_encoded = hi_tokenizer.encode_batch(df['hi'].tolist())

        print("Building tensors...")
        self.src_data = [
            torch.tensor([en_sos] + e.ids + [en_eos], dtype=torch.long)
            for e in en_encoded
        ]
        self.tgt_data = [
            torch.tensor([hi_sos] + h.ids + [hi_eos], dtype=torch.long)
            for h in hi_encoded
        ]
        print(f"Done: {len(self.src_data)} pairs ready")
        
    def __len__(self):
        return len(self.src_data)
        
    def __getitem__(self,index):
        
        return self.src_data[index], self.tgt_data[index]

In [54]:
from torch.nn.utils.rnn import pad_sequence

# adding padding according to the current batch
def collate_fn(batch):
    src,trg = zip(*batch)
    src_padded = pad_sequence(src,batch_first=True,padding_value=0)
    trg_padded = pad_sequence(trg,batch_first=True,padding_value=0)
    src_mask   = (src_padded != 0)
    return src_padded,trg_padded,src_mask

In [55]:
BATCH_SIZE=128

# Making Datasets 
train_dataset = TranslationDataset(train_df,en_tokenizer,hi_tokenizer)
val_dataset = TranslationDataset(val_df,en_tokenizer,hi_tokenizer)
test_dataset = TranslationDataset(test_df,en_tokenizer,hi_tokenizer)

# Making DataLoaders 
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_fn,pin_memory=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn,pin_memory=True)

print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Pre-tokenizing English...
Pre-tokenizing Hindi...
Building tensors...
Done: 108405 pairs ready
Pre-tokenizing English...
Pre-tokenizing Hindi...
Building tensors...
Done: 13536 pairs ready
Pre-tokenizing English...
Pre-tokenizing Hindi...
Building tensors...
Done: 13550 pairs ready
Train batches:      847
Validation batches: 106
Test batches:       106


In [56]:
class Encoder(nn.Module):
    
    def __init__(self,en_vocab_size,embed_dim=256,hidden_size=512,num_layers=2):
        super().__init__()
        
        self.embedding = nn.Embedding(en_vocab_size,
                                      embed_dim,
                                      padding_idx=0)
        
        self.lstm = nn.LSTM(embed_dim,
                            hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=0.3)
        
        self.fc_hidden = nn.Linear(hidden_size*2,hidden_size)
        self.fc_cell = nn.Linear(hidden_size*2,hidden_size)
        
    def forward(self,src,src_mask=None):
        embedded = self.embedding(src)
        
        output,(hidden,cell) = self.lstm(embedded)
        
        hidden = torch.cat([hidden[-2],hidden[-1]],dim=1)
        hidden = torch.tanh(self.fc_hidden(hidden))
        hidden = hidden.unsqueeze(0)

        cell = torch.cat([cell[-2],cell[-1]],dim=1)
        cell = torch.tanh(self.fc_cell(cell))
        cell = cell.unsqueeze(0)

        return output,hidden,cell

In [57]:
class LuongAttention(nn.Module):
    
    def __init__(self,hidden_size):
        super().__init__()
        self.Wa = nn.Linear(hidden_size*2,hidden_size,bias=False)

    def forward(self,decoder_hidden,encoder_outputs,src_mask=None):
        encoder_projected = self.Wa(encoder_outputs)
        # (4,10,512)
        decoder_hidden_t =decoder_hidden.permute(0,2,1)
        # (4,512,1)
        score = torch.bmm(encoder_projected,decoder_hidden_t)
        # (4,10,512)X(4,512,1)=(4,10,1)
        score = score.squeeze(-1)
        if src_mask is not None:
            score = score.masked_fill(src_mask == 0, -1e9) 
        
        attention_weights = torch.softmax(score,dim=1)
        attention_weights = attention_weights.unsqueeze(-1)
        # (4,10,1) but sum of probability of each word in each sentence would be 1 
        attention_weights_t = attention_weights.permute(0,2,1)
        # (4,1,10)
        contex_vector = torch.bmm(attention_weights_t,encoder_outputs)
        # (4,1,10)X(4,10,1024) = (4,1,1024)
        return contex_vector,attention_weights


In [58]:
# Quick shape test 
HIDDEN_SIZE = 512
BATCH_SIZE  = 4
SRC_LEN     = 10

dummy_decoder_hidden   = torch.randn(BATCH_SIZE, 1, HIDDEN_SIZE)
dummy_encoder_outputs  = torch.randn(BATCH_SIZE, SRC_LEN, HIDDEN_SIZE * 2)

attention = LuongAttention(HIDDEN_SIZE)
context, weights = attention(dummy_decoder_hidden, dummy_encoder_outputs)

print(f"Context vector shape:    {context.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Weights sum to 1: {weights.squeeze(-1).sum(dim=1)}")

Context vector shape:    torch.Size([4, 1, 1024])
Attention weights shape: torch.Size([4, 10, 1])
Weights sum to 1: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [59]:
class Decoder(nn.Module):
    
    def __init__(self,hi_vocab_size,embed_dim=256,hidden_size=512,num_layers=1,dropout=0.3):
        super().__init__()

        self.hidden_size=hidden_size
        self.hi_vocab_size=hi_vocab_size
        
        self.embedding = nn.Embedding(hi_vocab_size,embed_dim,padding_idx=0)
        self.lstm = nn.LSTM(embed_dim + hidden_size * 2,
                            hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=False,
                            dropout = dropout if num_layers > 1 else 0,
                           )
        self.attention = LuongAttention(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(hidden_size + hidden_size*2,hi_vocab_size)

    def forward(self,tgt_token,decoder_hidden,decoder_cell,encoder_outputs,src_mask=None):
            
        tgt_token = tgt_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt_token))
                
        decoder_hidden_for_att = decoder_hidden.permute(1,0,2)
        contex_vector,attention_weights = self.attention(decoder_hidden_for_att,encoder_outputs,src_mask)
            
        lstm_input = torch.cat([embedded,contex_vector],dim=2)
        lstm_output,(decoder_hidden,decoder_cell) = self.lstm(lstm_input,(decoder_hidden,decoder_cell))
            
        lstm_output = lstm_output.squeeze(1)
        contex_vector = contex_vector.squeeze(1)
        combine = torch.cat([lstm_output,contex_vector],dim=1)
            
        prediction = self.fc_out(combine)
            
        return prediction,decoder_hidden,decoder_cell,attention_weights
            
        

In [60]:
class Seq2Seq(nn.Module):
    
    def __init__(self,encoder,decoder,device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device 

    def forward(self,src,trg,src_mask=None,teacher_forcing_ratio=0.5):
        
        batch_size = src.shape[0]
        trg_seq_len = trg.shape[1]
        hi_vocab_size = self.decoder.hi_vocab_size

        all_prediction = torch.zeros(trg_seq_len,batch_size,hi_vocab_size).to(self.device)

        encoder_outputs,hidden,cell = self.encoder(src)
        decoder_input =trg[:,0]

        for t in range(1,trg_seq_len):
            prediction,hidden,cell,attention_weight = self.decoder(decoder_input,
                                                                    hidden,
                                                                    cell,
                                                                    encoder_outputs,
                                                                    src_mask
                                                                   )
            all_prediction[t] = prediction
            use_tech_forsing = torch.rand(1).item() < teacher_forcing_ratio

            if use_tech_forsing:
                decoder_input = trg[:,t]
            else:
                decoder_input = prediction.argmax(dim=1)

        return all_prediction



In [61]:
def initialize_weights(model):
    for name, param in model.named_parameters():
        if 'embedding' in name:
            nn.init.uniform_(param.data, -0.1, 0.1)
        elif 'weight_ih' in name:
            nn.init.xavier_uniform_(param.data)
        elif 'weight_hh' in name:
            nn.init.orthogonal_(param.data)
        elif 'weight' in name:
            nn.init.xavier_uniform_(param.data)
        elif 'bias' in name:
            nn.init.zeros_(param.data)

In [62]:
EN_VOCAB_SIZE = en_tokenizer.get_vocab_size()
HI_VOCAB_SIZE = hi_tokenizer.get_vocab_size()
EMBED_DIM = 256
HIDDEN_SIZE = 512
ENC_LAYERS = 2
DEC_LAYERS = 1
DROP_OUT = 0.3

encoder = Encoder(EN_VOCAB_SIZE,EMBED_DIM,HIDDEN_SIZE,ENC_LAYERS).to(device)
decoder = Decoder(HI_VOCAB_SIZE,EMBED_DIM,HIDDEN_SIZE,DEC_LAYERS,DROP_OUT).to(device)
model = Seq2Seq(encoder,decoder,device).to(device)

initialize_weights(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 31,093,568


In [63]:
import math
import time 


criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(),lr=0.0005)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

In [64]:
def train_one_epoch(model,loader,optimizer,criterion,clip=1.0,teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss=0

    for i,(src,trg,src_mask) in enumerate(loader):
        src=src.to(device)
        trg=trg.to(device)
        src_mask = src_mask.to(device)

        optimizer.zero_grad()
        
        output = model(src,trg,src_mask=src_mask,teacher_forcing_ratio=teacher_forcing_ratio)
        
        output_dim =output.shape[-1]
        output = output[1:].reshape(-1,output_dim)
        trg = trg[:,1:].reshape(-1)

        loss = criterion(output,trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),clip)

        optimizer.step()
        epoch_loss += loss.item()
        
        if (i + 1) % 100 == 0:
            print(f"  Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}")
            
    return epoch_loss/len(loader)
        

In [65]:
def evaluate(model,loader,criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src,trg,src_mask in loader:
            src = src.to(device)
            trg = trg.to(device)
            src_mask = src_mask.to(device)
            
            output = model(src,trg,src_mask=src_mask,teacher_forcing_ratio=0.0)
        
            output_dim =output.shape[-1]
            output = output[1:].reshape(-1,output_dim)
            trg = trg[:,1:].reshape(-1)

            loss = criterion(output,trg)
            epoch_loss += loss.item()
        
    return epoch_loss/len(loader)

In [ ]:
N_EPOCHS=20
best_val_loss = float('inf')

train_losses = []
val_losses = []

print(f"Starting training on {device}")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train PPL':>10} | {'Val PPL':>10} | {'Time':>8}")
print("-" * 65)

for epoch in range(1,N_EPOCHS+1):
    start_time = time.time()

    teacher_forcing_ratio=max(0.5,1.0- (epoch-1) * 0.05)
    train_loss = train_one_epoch(model,train_loader,optimizer,criterion,clip=1.0,teacher_forcing_ratio=teacher_forcing_ratio)
    val_loss = evaluate(model,val_loader,criterion)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    elapsed = time.time()- start_time

    train_ppl = math.exp(train_loss)
    val_ppl   = math.exp(val_loss)

    # Save best model whenever val loss improves
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        saved = "✓ saved"
    else:
        saved = ""

    print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>10.4f} | {train_ppl:>10.2f} | {val_ppl:>10.2f} | {elapsed:>6.1f}s  {saved}")

    # Safety checkpoint every 5 epochs
    # Protects against Kaggle session timeout
    if epoch % 5 == 0:
        torch.save({
            'epoch'          : epoch,
            'model_state'    : model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'train_losses'   : train_losses,
            'val_losses'     : val_losses,
            'best_val_loss'  : best_val_loss,
        }, f'/kaggle/working/checkpoint_epoch_{epoch}.pt')
        print(f"  → Safety checkpoint saved at epoch {epoch}")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f} | PPL: {math.exp(best_val_loss):.2f}")
    

Starting training on cuda
Epoch | Train Loss |   Val Loss |  Train PPL |    Val PPL |     Time
-----------------------------------------------------------------
  Batch 100/847 | Loss: 6.8777
  Batch 200/847 | Loss: 6.7678
  Batch 300/847 | Loss: 6.9064
  Batch 400/847 | Loss: 6.8260
  Batch 500/847 | Loss: 6.8531
  Batch 600/847 | Loss: 6.8926
  Batch 700/847 | Loss: 6.8903
  Batch 800/847 | Loss: 6.8565
    1 |     6.8972 |     6.8632 |     989.51 |     956.47 |  420.9s  ✓ saved
  Batch 100/847 | Loss: 6.8263
  Batch 200/847 | Loss: 6.8057
  Batch 300/847 | Loss: 6.8176
  Batch 400/847 | Loss: 6.8178
  Batch 500/847 | Loss: 6.8730
  Batch 600/847 | Loss: 6.9270
  Batch 700/847 | Loss: 6.8805
  Batch 800/847 | Loss: 6.8024
    2 |     6.8548 |     6.8579 |     948.42 |     951.37 |  419.9s  ✓ saved
  Batch 100/847 | Loss: 6.8386
  Batch 200/847 | Loss: 6.8519
  Batch 300/847 | Loss: 6.8392
  Batch 400/847 | Loss: 6.8851


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, 'b-o', label='Train Loss', markersize=4)
plt.plot(epochs, val_losses,   'r-o', label='Val Loss',   markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train vs Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, [math.exp(l) for l in train_losses], 'b-o', label='Train PPL', markersize=4)
plt.plot(epochs, [math.exp(l) for l in val_losses],   'r-o', label='Val PPL',   markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.title('Train vs Validation Perplexity')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print("Training curves saved")